# Tutorial: Embed plumes into an EMIT granule

Audience:
- Anyone who needs synthetic trace-gas plumes in real EMIT radiance
  for their own retrieval, detection or training pipeline.

Prerequisites:
- an EMIT L1B RAD granule and its OBS granule (collection 001);
- a `PlumeColumn` file of an LES plume (`spectra validate PLUME --contract plume-column`);
- the libRadtran/ARTS resource files and `spectra` installed.

Learning goals:
- describe an embedding run (scene files, plumes, forward model);
- run it and find what changed in the native output file;
- read the embedding metadata stored in that file.

SPECTRA writes a byte copy of the radiance file in which only the plume pixels
of the modelled bands change; the OBS granule applies to the output
unchanged. Paths below are placeholders: replace them with your files.

## Outline

1. Setup
2. Open the scene
3. Describe the run
4. Embed
5. What changed in the radiance
6. The embedding metadata
7. Everything else is untouched
8. Exercise, pitfalls and the command line

In [ ]:
# Setup: imports and the files of this tutorial (placeholders).
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import yaml

import spectra
from spectra.sensors import get_sensor

RADIANCE = Path("/path/to/EMIT_L1B_RAD_001_20260819T180247_2623112_032.nc")
OBS = Path("/path/to/EMIT_L1B_OBS_001_20260819T180247_2623112_032.nc")
PLUME = Path("/path/to/plume.plumecolumn.nc")  # PlumeColumn (MicroHH export)
RESOURCES = Path("/path/to/data-unit-abs-spec")  # libRadtran/ARTS bundle
OUTPUT = Path("/path/to/out/EMIT_L1B_RAD_001_20260819T180247_2623112_032_embedded.nc")
FILES = {"radiance": RADIANCE, "obs": OBS}

## Step 1 - Open the scene

The `emit` adapter opens the native files lazily as a `SensorScene`
(sensor geometry, native radiance units). This is only to look at the scene;
`spectra.embed` opens it itself.

In [ ]:
scene = get_sensor("emit").open_scene(FILES)
band = int(np.argmin(np.abs(scene["wavelength"].values - 2300.0)))
print(dict(scene.sizes), scene.attrs["scene_id"], scene.attrs["crs"])
print(f"quicklook band {band} at {float(scene['wavelength'][band]):.1f} nm")
scene["radiance"].isel(band=band)[::4, ::4].plot.imshow(
    x="across", y="along", robust=True, size=4
);

Expected result: the scene sizes, its UTM CRS and a decimated quicklook at
about 2300 nm, in the CH4 absorption window.

## Step 2 - Describe the run

A run is one scene and N plumes. Each plume is one snapshot (`time_index`) of
a plume file, anchored at a geographic position, a pixel or a seeded random
pixel; by default its emission source is pinned to the anchor. Adjust the
time indices to your plume file.

In [ ]:
# The libRadtran/ARTS forward model: every resource file is listed by role.
forward_model = {
    "solar_reference": str(
        RESOURCES
        / "spectrum/hybrid_reference_spectrum_p005nm_resolution_c2021-03-04_with_unc.nc"
    ),
    "tau": {
        name: str(
            RESOURCES
            / f"spectrum/UVSPEC_moltau_midlatitude-summer_{name}_1000-2500nm-0.001nm.nc"
        )
        for name in ("all", "CH4")
    },
    "profiles": {
        name: str(RESOURCES / f"arts/midlatitude-summer/midlatitude-summer.{name}.xml")
        for name in ("z", "t", "CH4")
    },
    "backgrounds": {"CH4": 1.93},  # ppm
    "scenario": "midlatitude-summer",
    "enhancement_max_ppm": 2000,
}

In [ ]:
lon = float(scene["lon"].isel(across=600, along=640))
lat = float(scene["lat"].isel(across=600, along=640))
run = {
    "sensor": "emit",
    "files": {role: str(path) for role, path in FILES.items()},
    "output": str(OUTPUT),
    "overwrite": True,
    "seed": 2026,
    "forward_model": forward_model,
    "plumes": [
        {"file": str(PLUME), "time_index": 1, "anchor": {"lon": lon, "lat": lat}, "rotation": 30},
        {
            "file": str(PLUME),
            "time_index": 1,
            "anchor": {"along": 645, "across": 610},
            "flip": "x",
            "emission_scale": 0.5,
        },
        {"file": str(PLUME), "time_index": 0, "anchor": "random"},
    ],
}
print(yaml.safe_dump(run["plumes"], sort_keys=False))

## Step 3 - Embed

Every plume is placed and regridded onto the pixel grid (mass-conserving),
plumes with overlapping windows form a cluster, each cluster gets one
absorption table from the forward model (about a minute and a half for 200
bands), and the native file is written.

In [ ]:
outcome = spectra.embed(run)
print(outcome.output)
for source in outcome.result.sources:
    print(
        f"plume {source.index}: cluster {source.cluster}, {source.anchor_mode} anchor, "
        f"{source.injected_mass_kg:.1f} kg on the grid, window {source.window}"
    )

Expected result: the first two plumes share a cluster when their windows
overlap; the random one usually forms its own.

## Step 4 - What changed in the radiance

The ratio of embedded to original radiance is below one where the plume
absorbs and exactly one elsewhere. The injected column of each plume is
stored on the `source` dimension; its sum is what was attenuated.

In [ ]:
original = xr.open_dataset(RADIANCE)
embedded = xr.open_dataset(OUTPUT)
window = outcome.result.clusters[0].window
native = {"crosstrack": window.across, "downtrack": window.along}
truth = embedded["injected_ch4_mass"].sum("source").isel(native)
ratio = (embedded["radiance"] / original["radiance"]).isel(native).isel(
    {"bands": band}
)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
truth.plot.imshow(ax=axes[0], cmap="magma")
axes[0].set_title("injected CH4 column (kg m-2)")
ratio.plot.imshow(ax=axes[1], cmap="viridis", vmax=1.0)
axes[1].set_title("embedded / original radiance")
plt.tight_layout()

In [ ]:
# The absorption spectrum at the strongest pixel: one below the modelled range.
peak = embedded["injected_ch4_mass"].sum("source").argmax(...)
spectrum = (embedded["radiance"] / original["radiance"]).isel(peak)
wavelength = scene["wavelength"].values
plt.figure(figsize=(8, 3))
plt.plot(wavelength, spectrum.values)
plt.xlabel("wavelength (nm)")
plt.ylabel("embedded / original")
plt.title("transmittance at the plume peak");

## Step 5 - The embedding metadata

Per-plume provenance lives in `source_*` variables, the run provenance in
`spectra_*` attributes, including the resolved run file (seed and drawn
anchors filled in), which reproduces the output.

In [ ]:
sources = embedded[[name for name in embedded.data_vars if name.startswith("source_")]]
sources.to_dataframe().T

In [ ]:
attrs = {key: value for key, value in embedded.attrs.items() if key.startswith("spectra_")}
print({key: attrs[key] for key in ("spectra_version", "spectra_gas", "spectra_seed")})
print(attrs["spectra_run_config"][:800])

## Step 6 - Everything else is untouched

Pixels outside the plume clusters, bands outside the modelled range and all
other variables keep their values (and bytes).

In [ ]:
outside = {"downtrack": slice(0, 50), "crosstrack": slice(0, 50)}
far = original["radiance"].isel(outside).values
print("far from the plumes identical:", np.array_equal(far, embedded["radiance"].isel(outside).values, equal_nan=True))
for name in ["flat_field_update"]:
    print(name, "identical:", bool(original[name].equals(embedded[name])))

## Exercise

Scale the emission of the first plume by three, write a second output and
compare the injected mass and the strongest attenuation with the first run.
Predict the result before running it.

In [ ]:
# Exercise answer scaffold
stronger = yaml.safe_load(embedded.attrs["spectra_run_config"])
stronger["plumes"][0]["emission_scale"] = 3.0
stronger["output"] = str(OUTPUT.with_name(OUTPUT.stem + "_x3.nc"))
second = spectra.embed(stronger)
for first, again in zip(outcome.result.sources, second.result.sources):
    print(first.index, f"{first.injected_mass_kg:.1f} kg -> {again.injected_mass_kg:.1f} kg")

## Pitfalls and extensions

- *Plume beyond the absorption table*: a strong plume core stops the run
  before the forward model runs and names the `enhancement_max_ppm` it
  needs; raise it in `forward_model`.
- *Existing output*: outputs are never overwritten unless `overwrite` is set,
  and an embedded file is refused as input, so plumes are never embedded
  twice.
- *Extension*: save the run as YAML and use the command line,
  `spectra embed run.yaml --dry-run` prints the resolved run without running
  the forward model.

In [ ]:
Path("run.yaml").write_text(yaml.safe_dump(run, sort_keys=False))
# !spectra embed run.yaml --dry-run